# Generation of new return period flood maps based on % change in peak flow

### Step 0: Import packages to work with and set up folder pathways

In [ ]:
from pathlib import Path
import numpy as np
import pandas
import rasterio
import scipy
from rasterio.warp import calculate_default_transform, reproject, Resampling

#### Output path

In [ ]:
output_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data")

#### Input paths

In [ ]:
# Import forests
forest_catchments_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/catchment_forest_summary_with_percentages.csv")
forest_catchments = pandas.read_csv(forest_catchments_path)

In [ ]:
forest_catchments.head()

In [ ]:
# Load interpolated peak flow reduction data
peak_flow_catchment_coverage_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/interpolated_peak_flow_catchment_coverage.csv")
peak_flow_catchment_coverage = pandas.read_csv(peak_flow_catchment_coverage_path)

In [ ]:
hazards_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Processed_data/Hazards")

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

### Step 1: Read in return period maps

In [ ]:
# List all files in the directory to check that it is pointing to the right place
all_files = list(hazards_path.iterdir())
display("All files in directory:", [f.name for f in all_files])

In [ ]:
# Read in river flood maps: found via processed data -> hazards -> Global Flood Map -> Jamaica -> Fluvial -> Raw Depths
# Tifs as follows: [Q20_RD_02.tif; Q50_RD_02.tif; Q100_RD_02.tif; Q200_RD_02.tif; Q500_RD_02.tif]  

river_flood_tiff_files = list(hazards_path.glob('*.tif'))

# Define the function to read a TIFF file
def read_rp_map(fname: Path) -> np.ndarray:
    """Read flood map TIFF file into a NumPy array."""
    with rasterio.open(fname) as dataset:
        data = dataset.read(1)
        data[data == dataset.nodata] = 0  # Replace no-data values with 0
    return data
    
# Read each TIFF file and store the data in a dictionary
river_flood_maps = {file.stem: read_rp_map(file) for file in river_flood_tiff_files}

# Print the names of the loaded files to verify - it prints the names of all the loaded tif files
display("Loaded flood maps:", river_flood_maps.keys())

### Step 2: Read in the file metadata

In [ ]:
# Read metadata from an existing GeoTIFF file specified by fname. Extract the following:
# -- CRS: The coordinate reference system.
# -- Width and Height: The raster’s dimensions (in pixels).
# -- Transform: The affine transformation matrix for georeferencing (e.g., linking pixel coordinates to geographic space).
# -- Purpose: This function does not modify the TIFF file; it only reads and returns the metadata.

def read_metadata(fname):
    with rasterio.open(fname) as dataset:
        crs = dataset.crs
        width = dataset.width
        height = dataset.height
        transform = dataset.transform
    return crs, width, height, transform

def reproject_raster(input_path, output_path, target_crs, target_shape=None, target_transform=None):
    """Reproject and resample a raster to a common CRS, resolution, and extent."""
    with rasterio.open(input_path) as src:
        if target_transform and target_shape:
            transform = target_transform
            width, height = target_shape
        else:
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )
        
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height,
        })
        
        with rasterio.open(output_path, 'w', **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=Resampling.nearest
                )

In [ ]:
# Save the data (a NumPy array) to a new GeoTIFF file at the specified fname. It includes the provided:
# -- CRS: Specifies the coordinate reference system for georeferencing.
# -- Transform: Links the raster data to geographic coordinates.
# -- Uses "w" mode to write a new file. If fname already exists, it will be overwritten.
# -- Compresses the data using LZW compression to save disk space.

def save_to_tif(data, fname, crs, transform):
    with rasterio.open(
        fname,
        "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=crs,
        transform=transform,
        compress="lzw",
    ) as dataset:
        dataset.write(data, 1)

### Step 3: Interpolate between flood maps in order to develop tifs of missing return periods (RP2,5,10)

In [ ]:
# Read all the different return period maps as TIFF files and return their data as a dictionary of NumPy arrays.

# Function Signature:
  # Input:
	# - rp_maps: A dictionary where:
	# - - Keys are float values representing return periods (e.g., 2, 5, 10, etc.).
	# - - Values are Path objects representing the file paths to the corresponding flood map TIFF files.
  # Output:
	# - A dictionary (rp_data) where:
	# - - Keys are the same return periods (float values).
	# - - Values are NumPy arrays (np.ndarray) containing the raster data from the corresponding TIFF files.

def read_rp_maps(rp_maps: dict[float, Path]) -> dict[float, np.ndarray]:
    """Read flood map TIFFs to dict of ndarrays, validating file existence"""
    rp_data: dict[float, np.ndarray] = {}
    for rp, fname in rp_maps.items():
        if not Path(fname).exists():
            raise FileNotFoundError(f"Input file not found: {fname}")
        rp_data[rp] = read_rp_map(fname)
    return rp_data

In [ ]:
def interpolate_depth(rp: float, rp_l: float, depth_l: np.ndarray, rp_u: float, depth_u: np.ndarray) -> np.ndarray:
    """Interpolate between two flood map ndarrays
    """
    rp_factor = (np.log(rp) - np.log(rp_l)) / (np.log(rp_u) - np.log(rp_l))
    depth = depth_l + ((depth_u - depth_l) * rp_factor)

    return depth

In [ ]:
def pick_upper_lower_rps(rp: float, rps: list[float]) -> tuple[float]:
    bin_index = np.searchsorted(rps, rp, side="left")
    rp_l = rps[bin_index - 1]
    rp_u = rps[bin_index]
    return rp_l, rp_u

In [ ]:
def calculate_rp_maps(rps_to_calculate, rps_input):
    # Read all data
    depths = read_rp_maps(rps_input)
    baseline_rps = list(depths.keys())
    # Read metadata
    ## The 1e-3 I think means that it is not quite 0, just above it. Hence possibly why we get the 0.001 as an output of the next box.
    crs, width, height, transform = read_metadata(rps_input[baseline_rps[0]])
    depths[1e-3] = np.zeros((height, width))
    depths[2.0] = np.zeros((height, width))
    depths[1e6] = depths[max(baseline_rps)]
    baseline_rps = [1e-3, 2] + sorted(baseline_rps) + [1e6]

# Calculate and save interpolated depths for new return periods
    for rp, output_fname in rps_to_calculate.items():
        rp_l, rp_u = pick_upper_lower_rps(rp, baseline_rps)
        print(rp_l, rp_u)
        depth = interpolate_depth(rp, rp_l, depths[rp_l], rp_u, depths[rp_u])

# Save the output depth map to the specified file
        save_to_tif(depth, output_fname, crs, transform)

In [ ]:
# Input return periods and file paths
rps_input = {
    20: hazards_path / "JM_FLRF_UD_Q20_RD_02.tif",
    50: hazards_path / "JM_FLRF_UD_Q50_RD_02.tif",
    100: hazards_path / "JM_FLRF_UD_Q100_RD_02.tif",
    200: hazards_path / "JM_FLRF_UD_Q200_RD_02.tif",
    500: hazards_path / "JM_FLRF_UD_Q500_RD_02.tif"
}

# Define output return periods
rps_to_calculate = {
    2: hazards_path / "JM_FLRF_UD_Q2_RD_02_aligned.tif",
    5: hazards_path / "JM_FLRF_UD_Q5_RD_02_aligned.tif",
    10: hazards_path / "JM_FLRF_UD_Q10_RD_02_aligned.tif"
}

# Define the reference raster
reference_raster = hazards_path / "JM_FLRF_UD_Q20_RD_02.tif"

# Reproject and align all input rasters
aligned_rasters = {}
with rasterio.open(reference_raster) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_shape = (ref.height, ref.width)
    
    for rp, raster_path in rps_input.items():
        hazards_output_path = hazards_path / f"{raster_path.stem}_aligned.tif"
        reproject_raster(raster_path, hazards_output_path, ref_crs, ref_shape, ref_transform)
        aligned_rasters[rp] = hazards_output_path

# Update rps_input to use aligned rasters
rps_input = aligned_rasters

# Call the calculate_rp_maps function
calculate_rp_maps(rps_to_calculate, rps_input)

rp_data = read_rp_maps(rps_input)

In [ ]:
print("Loaded return periods:", rp_data.keys())
print("Data for RP 20:", rp_data[20])  # Print the NumPy array for RP 20

In [ ]:
print("Checking input files...")
for rp, path in rps_input.items():
    if not Path(path).exists():
        print(f"Input file not found: {path}")

In [ ]:
hazards_output_file = hazards_path / "JM_FLRF_UD_Q2_RD_02.tif"
with rasterio.open(hazards_output_file) as dataset:
    data = dataset.read(1)
    print("Data for RP 2:", data)

In [ ]:
print("Expected output files:")
for rp, path in rps_to_calculate.items():
    print(f"{rp}: {path}")

### Step 4: Define how return period changes according to percentage change in peak flow

#### Interpolate to a full table of catchment forest percentages and peak flow reductions for RP5 and 100

In [ ]:
# Create the initial dataframe with the original column name
peak_flow_catchment_coverage = pandas.DataFrame({
    'catchment_forest_percentage': [0, 6, 14, 21, 62, 100],
    'rp5.0': [0, 3, 13, 18, 48, 48],
    'rp100.0': [0, 1, 8, 11, 32, 32],
})
# peak_flow_catchment_coverage.rename(columns={'catchment_forest_percentage': 'Catchment forest coverage (%)'}, inplace=True)
display(peak_flow_catchment_coverage)

# Create a full range of percentages from 0 to 100
full_percentages = pandas.DataFrame({'catchment_forest_percentage': np.arange(0, 101)})

# Merge the full range with the existing data
interpolated_data = pandas.merge(full_percentages, peak_flow_catchment_coverage, on='catchment_forest_percentage', how='left')

# Perform linear interpolation to fill missing values
interpolated_data['rp5.0'] = interpolated_data['rp5.0'].interpolate(method='linear')
interpolated_data['rp100.0'] = interpolated_data['rp100.0'].interpolate(method='linear')

# interpolated_data.rename(columns={'catchment_forest_percentage': 'Catchment forest coverage (%)'}, inplace=True)
interpolated_data = interpolated_data.round(2)
interpolated_data.to_csv("interpolated_peak_flow_catchment_coverage.csv", index=False)
display(interpolated_data)

In [ ]:
def get_rp_cols(df):
    rp_cols = [col for col in df.columns if "rp" in col]
    rps = [float(col.replace("rp", "")) for col in rp_cols]
    return rp_cols, rps

def peak_flow_reduction(forest_perc, lookup):
    data = lookup.catchment_forest_percentage
    rp_cols, rps = get_rp_cols(lookup)
    interpolator = scipy.interpolate.RegularGridInterpolator(
        (rps, data),
        lookup[rp_cols].values.T,
        method='linear'
    )
    vals = interpolator(([rps], [forest_perc]))[0]
    return pandas.DataFrame({
        'catchment_forest_percentage': [forest_perc],
        'rp5.0': vals[0],
        'rp100.0': vals[1],
    })

peak_flow_reduction(7, peak_flow_catchment_coverage)

#### % Change in peak flow impact on return period

In [ ]:
# Top row outlines change in peak flow. Subsequent rows define the return period and what it becomes.
### Assume for return period of 2 or below, flood depth is 0, i.e. the infrastructure asset experiences no damage. 

""
"Need to add in 30% values - is there a way to interpolate this?"
""

CCRA_flow_reductions = pandas.DataFrame({
    'reduction_percent': [5, 10, 20, 40],
    #'rp2.0': [2.4, 3.2, 6.8, 196],
    #'rp2.3': [2.8, 3.7, 7.9, 169],
    'rp5.0': [6.5, 8.9, 19, 235],
    'rp10.0': [13,19,43,473],
    #'rp25.0': [35, 51, 123, 1330],
    'rp50.0': [72,107,268,2967],
    'rp100.0': [147,224,582, 6648],
    #'rp500.0': [772,1229,3441,43094],
    #'rp1000.0': [1571,2544,7345,95943],
})

# proportion of baseline flow
CCRA_flow_reductions['flow'] = (1 - CCRA_flow_reductions.reduction_percent / 100)

# The below code is used to interpolate based on the current return periods (2,12,50,100,500,1000) to define a new return period (20)
## interpolate RP20 values
known_rp_cols = [rp_col for rp_col in CCRA_flow_reductions.columns if "rp" in rp_col]
known_rps = [float(rp.replace("rp","")) for rp in known_rp_cols]
flows = CCRA_flow_reductions['flow'].values

interpolator = scipy.interpolate.RegularGridInterpolator(
    (known_rps, flows),
    CCRA_flow_reductions[known_rp_cols].values.T,
    method='cubic'
)

rps_new = [[20.0]]
CCRA_flow_reductions['rp20.0'] = interpolator((rps_new, [flows]))[0]

out_rps = sorted([20.0] + known_rps)
out_rp_cols = [f"rp{rp}" for rp in out_rps]

CCRA_flow_reductions_interpolated = CCRA_flow_reductions[['flow'] + out_rp_cols].copy()

flow07_row_values = interpolator(([out_rps], [[0.7]]))[0]
flow07_row = pandas.DataFrame(data=[[0.7] + list(flow07_row_values)], columns=['flow']+out_rp_cols)

CCRA_flow_reductions_interpolated = (
    pandas.concat([CCRA_flow_reductions_interpolated, flow07_row])
    .sort_values("flow", ascending=False)
    .reset_index(drop=True)
)

### Forest catchment coverage

In [ ]:
def interpolate_flow_reductions(current_cover_perc, future_cover_perc, flow_reductions_df, return_period):
    """
    Interpolates peak flow reductions based on current and future forest cover percentages.
    
    Parameters:
        current_cover_perc (float): The current forest cover percentage.
        future_cover_perc (float): The future forest cover percentage (e.g., after reforestation).
        flow_reductions_df (pd.DataFrame): DataFrame containing forest cover percentages and flow reductions.
        return_period (str): The column name in the DataFrame for the return period (e.g., 'rp5.0').
    
    Returns:
        tuple: (current_flow_reduction, future_flow_reduction) interpolated for the given return period.
    """
    # Ensure percentages are within bounds
    current_cover_perc = max(0, min(100, current_cover_perc))
    future_cover_perc = max(0, min(100, future_cover_perc))
    
    # Interpolate flow reductions for the current and future percentages
    current_flow_reduction = np.interp(
        current_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    future_flow_reduction = np.interp(
        future_cover_perc,
        flow_reductions_df['catchment_forest_percentage'],
        flow_reductions_df[return_period]
    )
    
    return current_flow_reduction, future_flow_reduction

# Define return periods to analyze
return_periods = ['rp5.0', 'rp100.0']

# Loop through each return period and calculate reductions
for return_period in return_periods:
    # 1) Interpolate current & future flow reductions
    forest_catchments[f'current_flow_reduction_{return_period}'], forest_catchments[f'future_flow_reduction_{return_period}'] = zip(*forest_catchments.apply(
        lambda row: interpolate_flow_reductions(
            current_cover_perc=row['forest_flood_equivalent_percentage'],
            future_cover_perc=row['total_future_forest_including_agri_percentage'],
            flow_reductions_df=peak_flow_catchment_coverage,
            return_period=return_period
        ),
        axis=1
    ))
    
    # 2) Compute the ratio-based columns
    forest_catchments[f'future_ratio_{return_period}'] = 100 - forest_catchments[f'future_flow_reduction_{return_period}']
    forest_catchments[f'current_ratio_{return_period}'] = 100 - forest_catchments[f'current_flow_reduction_{return_period}']
    
    forest_catchments[f'change_in_ratio_{return_period}'] = (
        forest_catchments[f'current_ratio_{return_period}'] 
        - forest_catchments[f'future_ratio_{return_period}']
    )
    
    forest_catchments[f'future_reduction_proportion_{return_period}'] = (
        forest_catchments[f'change_in_ratio_{return_period}'] 
        / forest_catchments[f'current_ratio_{return_period}']
    ) * 100

# Now do rounding
columns_to_round = [col for col in forest_catchments.columns if 'flow_reduction' in col or 'reduction_difference' in col]
columns_to_round += [col for col in forest_catchments.columns if 'ratio' in col or 'future_reduction_proportion' in col]
forest_catchments[columns_to_round] = forest_catchments[columns_to_round].round(2)

# Debug
display(
    forest_catchments[
        [
            'HYBAS_ID', 
            'forest_flood_equivalent_percentage', 
            'total_future_forest_including_agri_percentage'
        ] 
        + columns_to_round
    ].head()
)

# Export updated data to CSV for each return period
for return_period in return_periods:
    output_ratio_csv_path = output_path / f"peak_flow_reduction_ratios_{return_period}_by_hydrobasin.csv"
    forest_catchments.to_csv(output_ratio_csv_path, index=False)
    print(f"Data with ratios successfully exported to {output_ratio_csv_path}")

In [ ]:
display(CCRA_flow_reductions_interpolated)

In [ ]:
for _, row in CCRA_flow_reductions.iterrows():
    flow = row.flow
    
    # declare desired output RP and filename
    rps_to_calculate = {
        10: hazards_path / f"JM_FLRF_UD_Q10_RD_02_flow{flow}.tif",
        20: hazards_path / f"JM_FLRF_UD_Q20_RD_02_flow{flow}.tif",
        50: hazards_path / f"JM_FLRF_UD_Q50_RD_02_flow{flow}.tif",
        100: hazards_path / f"JM_FLRF_UD_Q100_RD_02_flow{flow}.tif",
    }
    # use baseline RP maps "mislabelled" as after change in flow
    rps_input = {
        #row.rp2: hazards_path / "JM_FLRF_UD_Q2_RD_02_aligned.tif",
        row['rp10.0']: hazards_path / "JM_FLRF_UD_Q10_RD_02_aligned.tif",
        row['rp20.0']: hazards_path / "JM_FLRF_UD_Q20_RD_02_aligned.tif",
        row['rp50.0']: hazards_path / "JM_FLRF_UD_Q50_RD_02_aligned.tif",
        row['rp100.0']: hazards_path / "JM_FLRF_UD_Q100_RD_02_aligned.tif",
    }
    calculate_rp_maps(rps_to_calculate, rps_input)

In [ ]:
for rp, path in {**rps_to_calculate, **rps_input}.items():
    if not Path(path).exists():
        print(f"File not found: {path}")

In [ ]:
print("All files in hazards_path:")
for file in hazards_path.iterdir():
    print(file.name)